In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import MarianMTModel, MarianTokenizer
import torch
from google.colab import drive
from google.colab import files

# First, check if GPU is available
print("Checking GPU availability...")
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

class CovidDataProcessor:
    def __init__(self, model_name="Helsinki-NLP/opus-mt-en-fr"):
        # Mount Google Drive
        drive.mount('/content/drive')

        # Initialize tokenizer and model
        self.tokenizer = MarianTokenizer.from_pretrained(model_name)
        self.model = MarianMTModel.from_pretrained(model_name)

        # Move model to GPU if available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.model.to(self.device)
        print(f"Using device: {self.device}")

    def translate_text(self, text, max_length=512):
        if pd.isna(text) or text.strip() == '':
            return text

        try:
            # Prepare input
            inputs = self.tokenizer(text, return_tensors="pt", truncation=True,
                                  max_length=max_length, padding=True)

            # Move input tensors to GPU
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            # Generate translation
            with torch.no_grad():
                outputs = self.model.generate(**inputs, max_length=max_length)

            # Move output back to CPU for decoding
            outputs = outputs.cpu()
            translation = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            return translation

        except Exception as e:
            print(f"Error translating text: {text}")
            print(f"Error message: {str(e)}")
            return text

    def translate_csv(self, input_path, output_path):
        print("Reading CSV file...")
        # Try different encodings
        encodings = ['utf-8', 'latin1', 'iso-8859-1', 'cp1252']

        df = None
        for encoding in encodings:
            try:
                # Read only first 29 lines
                df = pd.read_csv(input_path, encoding=encoding)
                print(f"Successfully read file with {encoding} encoding")
                break
            except UnicodeDecodeError:
                continue

        if df is None:
            raise Exception("Could not read the CSV file with any of the attempted encodings")

        print("Starting translation...")
        for column in df.columns:
            print(f"Translating {column}...")
            df[column] = df[column].astype(str).apply(
                lambda x: self.translate_text(x)
            )

        # Save to Google Drive with utf-8 encoding
        df.to_csv(output_path, index=False, encoding='utf-8')
        print(f"Translation completed! Saved to {output_path}")
        return df

    def analyze_data(self, df):
        print("\nDataset Info:")
        print(df.info())

        print("\nFirst few rows:")
        print(df.head())

        print("\nBasic statistics:")
        print(df.describe(include='all'))

        for column in df.columns:
            print(f"\nNumber of unique {column}s:", df[column].nunique())

        self.create_visualizations(df)

    def create_visualizations(self, df):
        plt.figure(figsize=(15, 10))

        # Question length distribution
        plt.subplot(2, 1, 1)
        question_lengths = df['Question'].str.len()
        sns.histplot(data=question_lengths, bins=20)
        plt.title('Distribution of Question Lengths')
        plt.xlabel('Length of Question')
        plt.ylabel('Count')

        # Answer length distribution
        plt.subplot(2, 1, 2)
        answer_lengths = df['Answer'].str.len()
        sns.histplot(data=answer_lengths, bins=20)
        plt.title('Distribution of Answer Lengths')
        plt.xlabel('Length of Answer')
        plt.ylabel('Count')

        plt.tight_layout()
        plt.savefig('/content/drive/My Drive/covid_analysis_plots.png')
        plt.close()

def setup_gpu():
    if torch.cuda.is_available():
        torch.cuda.set_device(0)
        print(f"Using GPU: {torch.cuda.get_device_name(0)}")
        print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/1024**2:.2f} MB")
        print(f"Memory Cached: {torch.cuda.memory_reserved(0)/1024**2:.2f} MB")
        torch.cuda.empty_cache()
        return True
    else:
        print("No GPU available, using CPU instead.")
        return False

def main():
    # Install required packages
    !pip install transformers torch pandas sacremoses

    # Setup GPU
    setup_gpu()

    # Initialize processor
    processor = CovidDataProcessor()

    # File paths in Google Drive
    input_path = '/content/drive/My Drive/covid_dataset.csv'
    output_path = '/content/drive/My Drive/covid_dataset_french.csv'

    try:
        print("Starting data processing...")
        df = processor.translate_csv(input_path, output_path)

        print("\nAnalyzing translated data...")
        processor.analyze_data(df)

    except FileNotFoundError:
        print(f"Error: Could not find the input file '{input_path}'")
    except Exception as e:
        print(f"An error occurred: {str(e)}")

    finally:
        # Clean up GPU memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if __name__ == "__main__":
    main()

Checking GPU availability...
CUDA Available: True
GPU Name: Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 17.3 MB/s eta 0:00:00
Using GPU: Tesla T4
Memory Allocated: 0.00 MB
Memory Cached: 0.00 MB
Mounted at /content/drive


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/301M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Using device: cuda
Starting data processing...
Reading CSV file...
Successfully read file with latin1 encoding
Starting translation...
Translating Answer...
Translating Question...
